## 02 Data Cleaning

#### Purpose
- Investigate the data sanity and quality 
- Clean the data

#### Steps: 
- DuckDB setup and memory limits
- Data-quality audit of the raw data (dtypes, statistics, true nulls and placeholder values, invalid codes, impossible values, constant columns)
- Cleaning rules: policies
- Cleaning rules: claims
- Cleaning rules: disasters, community, ACS, FRED
- Before/after counts and cleaning_log.csv
- Export cleaned parquet

#### Authors
Organized and prepared by Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III. 

- **Date Created:** 2026-11-07
- **Date Modified:** 2026-11-07

### Section 0. Load the libraries and working directory

In [36]:
# Import tools
import numpy as np
import pandas as pd
import scipy as sp
import os
from pathlib import Path
import re
import urllib.request
import os, json, urllib.parse
from dotenv import load_dotenv
import duckdb

# Set working directory
# Repo root = parent of notebooks/ (works on any machine)
working_directory = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_data_folder = working_directory /"data" / "raw"
interim_folder = working_directory / "data" / "interim"
processed_data_folder = working_directory /"data" /"processed"
reports_folder = working_directory / "reports"

data_dict = working_directory / "docs" / "data_dictionary"


### Section 1. Load the datasets
Our datasets include the following (all downloaded in full, unchanged, into data/raw/):
1. NFIP Redacted Policies v3: about 74.7M policy terms (OpenFEMA bulk Parquet)
2. NFIP Redacted Claims v3: about 2.7M claims (OpenFEMA bulk Parquet)
3. Disaster Declarations Summaries (OpenFEMA)
4. NFIP Community Status Book (OpenFEMA)
5. FRED macroeconomic series
6. Census ACS 5-year (2022), tract level

We also need to load the data dictionaries: 
1. fema_NfipCommunityStatusBook_v1.xlsx
2. census_acs5_2022_selected.xlsx
3. fema_DisasterDeclarationsSummaries_v2.xlsx
4. fema_NfipClaims_v3.xlsx
5. fema_NfipPolicies_v3.xlsx

Steps: 
- Create the folders data/interim/, data/processed/ and reports/.
- Open a DuckDB connection to a database file (data/interim/nfip.duckdb), not an in-memory one.
- Set the memory limit to about half your RAM, set a temp directory for spill files, and set preserve_insertion_order = false.
- Create views over the raw Parquet files: policies, claims, disasters, community, and ACS. Never edit the raw files.
- Load the data dictionary CSVs you saved. They’re your reference for every rule.
- Create a cleaning_log list in Python. Every rule you apply adds one entry: table, column, rule, reason, and rows affected.

In [15]:
# Open a connection
con = duckdb.connect(
    str(interim_folder / "nfip.duckdb"),
    config={"preserve_insertion_order": False},
)

# Set parameters
con.execute("SET memory_limit = '8GB'") # set limit to RAM usage
con.execute(f"SET temp_directory = '{interim_folder / 'duckdb_tmp'}'")

# View of the raw files
fema = raw_data_folder / "fema"
raw_files = {
    "raw_policies": fema / "nfip_policies_v3.parquet",
    "raw_claims": fema / "nfip_claims_v3.parquet",
    "raw_community": fema / "nfip_community_status_book_v1.parquet",
    "raw_disasters": fema / "disaster_declarations_summaries_v2.parquet",
}

for view, path in raw_files.items():
    con.execute(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM read_parquet('{path}')")

con.sql("SHOW TABLES")

nameVARCHAR
clean_policies
raw_claims
raw_community
raw_disasters
raw_policies


In [25]:
# Load the datasets FRED
fred = raw_data_folder / "fred"

cpi = pd.read_csv(fred / "CPIAUCSL.csv")
csu = pd.read_csv(fred / "CSUSHPINSA.csv")
dgs = pd.read_csv(fred / "DGS10.csv")
fedfunds = pd.read_csv(fred / "FEDFUNDS.csv")
mortgage = pd.read_csv(fred / "MORTGAGE30US.csv")
unrate = pd.read_csv(fred / "UNRATE.csv")

In [26]:
# Load the datasets from Census
census = raw_data_folder / "census"

acs =pd.read_csv(census / "acs5_2022_tract.csv")
acs.head(5).T

,0,1,2,3,4
NAME,Census Tract 2.01; Alachua County; Florida,Census Tract 2.02; Alachua County; Florida,Census Tract 3.01; Alachua County; Florida,Census Tract 3.02; Alachua County; Florida,Census Tract 4; Alachua County; Florida
median_household_income,17727,13045,45227,41868,50417
median_home_value,-666666666,235100,211400,152200,161100
median_age,21.5,20.9,26.7,37.4,29.7
occupied_housing_units,1960,1690,1816,1237,2242
owner_occupied_units,151,301,468,670,1321
total_population,4856,5464,4034,2363,5526
state,12,12,12,12,12
county,1,1,1,1,1
tract,201,202,301,302,400


In [35]:
# Create a clean log
cleaning_log = []
def log_rule(table, column, rule, reason, rows_affected):
    cleaning_log.append({"table": table, "column": column, "rule": rule, 
                         "reason": reason, "rows_affected": rows_affected})

In [41]:
# Load the data dictionaries
claim_dict = pd.read_excel(data_dict / "fema_NfipClaims_v3.xlsx")
policies_dict = pd.read_excel(data_dict / "fema_NfipPolicies_v3.xlsx")
disaster_dict = pd.read_excel(data_dict / "fema_DisasterDeclarationsSummaries_v2.xlsx")
census_dict = pd.read_excel(data_dict / "census_acs5_2022_selected.xlsx")
community_dict = pd.read_excel(data_dict / "fema_NfipCommunityStatusBook_v1.xlsx")

### Section 2. Data Quality Audit
- Do data quality and sanity check

Steps: 
- DESCRIBE: list the stored dtypes. Note dates or numbers stored as text.
- SUMMARIZE: min, max, null %, approximate distinct count.
- Compare each column with the dictionary: is the type and meaning what it  should be?
- For text columns, find placeholder values such as "None", "", "NA", "N/A", "0", "9999" and whitespace-only strings. Count each one.
- For coded columns (zones, occupancy, deductibles, reason codes), list the distinct values and compare them with the legal values in the dictionary.  Count the invalid ones.
- Flag constant or near-constant columns and columns that are 100% null.
- Check plausibility: negative premiums or coverage, coverage above NFIP limits, termination before effective date, construction dates in the future or before 1800, loss dates outside 1978 to today.
- Check uniqueness: duplicate ids, and fully duplicated rows apart from id.
- Check integrity: claim censusGeoid and zip codes that never appear in policies, and state/county mismatches.
- Save everything into one dq_audit_raw table: a row per column with % missing, % placeholder, % invalid and so on. This table is your evidence that the data is messy. Export it to reports/.

#### 2.1 Check the dtypes of each data

In [14]:
# Step 1a. Check the dtypes of the data from OpenFEMA
for view in ["raw_policies", "raw_claims", "raw_community", "raw_disasters"]:
    print(view)
    display(con.sql(f"DESCRIBE {view}").df()[["column_name", "column_type"]])

raw_policies


,column_name,column_type
0,agricultureStructureIndicator,BOOLEAN
1,asOfDate,TIMESTAMP WITH TIME ZONE
2,baseFloodElevation,"DECIMAL(8,2)"
3,basementEnclosureCrawlspaceType,SMALLINT
4,cancellationDateOfFloodPolicy,DATE
...,...,...
82,fullRiskPremium,INTEGER
83,buildingOverWaterType,SMALLINT
84,foundationType,VARCHAR
85,preFIRMConstructionIndicator,BOOLEAN


raw_claims


,column_name,column_type
0,agricultureStructureIndicator,BOOLEAN
1,asOfDate,TIMESTAMP WITH TIME ZONE
2,basementEnclosureCrawlspaceType,SMALLINT
3,policyCount,SMALLINT
4,crsClassCode,SMALLINT
...,...,...
79,totalBldgClaimPmtRecovery,"DECIMAL(12,2)"
80,totalContentsClaimPmtRecovery,"DECIMAL(12,2)"
81,totalIccClaimPmtRecovery,"DECIMAL(12,2)"
82,totalSubrogationRecovery,"DECIMAL(12,2)"


raw_community


,column_name,column_type
0,communityIdNumber,VARCHAR
1,communityName,VARCHAR
2,county,VARCHAR
3,state,VARCHAR
4,initialFloodHazardBoundaryMap,DATE
5,initialFloodInsuranceRateMap,DATE
6,currentlyEffectiveMapDate,VARCHAR
7,regularEmergencyProgramDate,VARCHAR
8,tribal,BOOLEAN
9,participatingInNFIP,BOOLEAN


raw_disasters


,column_name,column_type
0,id,VARCHAR
1,disasterNumber,SMALLINT
2,state,VARCHAR
3,femaDeclarationString,VARCHAR
4,declarationType,VARCHAR
5,declarationDate,DATE
6,fyDeclared,SMALLINT
7,incidentType,VARCHAR
8,declarationTitle,VARCHAR
9,ihProgramDeclared,BOOLEAN


In [33]:
# Step 1.b Check the dtypes of other tables
other_tables = {
    "acs": acs,
    "cpi": cpi,
    "dgs": dgs,
    "fedfunds": fedfunds,
    "mortgage": mortgage,
    "unrate": unrate,
}

for name, df in other_tables.items():
    print(name, df.shape)
    print(df.dtypes, "\n")

acs (5160, 10)
NAME                        object
median_household_income      int64
median_home_value            int64
median_age                 float64
occupied_housing_units       int64
owner_occupied_units         int64
total_population             int64
state                        int64
county                       int64
tract                        int64
dtype: object 

cpi (957, 2)
observation_date     object
CPIAUCSL            float64
dtype: object 

dgs (16918, 2)
observation_date     object
DGS10               float64
dtype: object 

fedfunds (868, 2)
observation_date     object
FEDFUNDS            float64
dtype: object 

mortgage (2902, 2)
observation_date     object
MORTGAGE30US        float64
dtype: object 

unrate (946, 2)
observation_date     object
UNRATE              float64
dtype: object 



#### 2.2 Summary Statistics

In [43]:
# Step 2a. Generate the summary statistics for each column
summaries = {}
for view in ["raw_policies", "raw_claims", "raw_community", "raw_disasters"]:
    s = con.sql(f"SUMMARIZE {view}").df()
    s["null_percentage"] = pd.to_numeric(s["null_percentage"].astype(str).str.rstrip("%"))
    s["null_count"] = (s["count"] * s["null_percentage"]/100).round().astype("int64")
    summaries[view] = s
    s.to_excel(reports_folder / f"summary_{view}.xlsx", index=False)

    print(view, f"({len(s)} columns, {s['count'].iloc[0]:,} rows)")
    display(s[["column_name", "column_type", "min", "max", "null_count", "null_percentage", "approx_unique", "avg", "std"]])

raw_policies (87 columns, 74,739,461 rows)


,column_name,column_type,min,max,null_count,null_percentage,approx_unique,avg,std
0,agricultureStructureIndicator,BOOLEAN,false,true,0,0.00,2,None,None
1,asOfDate,TIMESTAMP WITH TIME ZONE,2026-09-08 08:00:00+08,2026-09-08 08:00:00+08,0,0.00,1,2026-09-08 08:00:00+08,None
2,baseFloodElevation,"DECIMAL(8,2)",-9999.00,9998.00,50224918,67.20,37535,315.17848085592175,1463.4310083781697
3,basementEnclosureCrawlspaceType,SMALLINT,0,4,44290605,59.26,4,0.5993840765609939,1.0171175182580048
4,cancellationDateOfFloodPolicy,DATE,2003-10-20,2027-08-05,71585456,95.78,6515,2017-07-01 12:38:32.495659,None
...,...,...,...,...,...,...,...,...,...
82,fullRiskPremium,INTEGER,16,1420151,35501244,47.50,28865,1967.457459281744,4139.201071168308
83,buildingOverWaterType,SMALLINT,1,3,3901400,5.22,3,1.0022872552808162,0.06583248140322671
84,foundationType,VARCHAR,0,6,0,0.00,7,None,None
85,preFIRMConstructionIndicator,BOOLEAN,false,true,0,0.00,2,None,None


raw_claims (84 columns, 2,725,989 rows)


,column_name,column_type,min,max,null_count,null_percentage,approx_unique,avg,std
0,agricultureStructureIndicator,BOOLEAN,false,true,0,0.00,2,None,None
1,asOfDate,TIMESTAMP WITH TIME ZONE,2026-09-08 08:00:00+08,2026-09-08 08:00:00+08,0,0.00,1,2026-09-08 08:00:00+08,None
2,basementEnclosureCrawlspaceType,SMALLINT,0,4,1902195,69.78,4,1.1759783148354737,1.0658116135797049
3,policyCount,SMALLINT,1,1090,0,0.00,409,1.313024373906131,6.795334193690077
4,crsClassCode,SMALLINT,1,10,2174249,79.76,10,6.26336556499044,1.6912604278042178
...,...,...,...,...,...,...,...,...,...
79,totalBldgClaimPmtRecovery,"DECIMAL(12,2)",-20000.00,1000000.00,2689188,98.65,6479,2309.556603322187,14704.675104104477
80,totalContentsClaimPmtRecovery,"DECIMAL(12,2)",-19040.00,250000.00,2689188,98.65,2690,1566.2225210015497,4652.329644334026
81,totalIccClaimPmtRecovery,"DECIMAL(12,2)",-15000.00,30000.00,2689188,98.65,241,97.67650327596988,1200.9727871516923
82,totalSubrogationRecovery,"DECIMAL(12,2)",-2000.00,834621.46,2689188,98.65,298,119.67324851154065,4775.03404743268


raw_community (18 columns, 25,133 rows)


,column_name,column_type,min,max,null_count,null_percentage,approx_unique,avg,std
0,communityIdNumber,VARCHAR,010001,780000,0,0.00,25912,None,None
1,communityName,VARCHAR,ABBEVILLE COUNTY *,"ZWOLLE, TOWN OF",0,0.00,19376,None,None
2,county,VARCHAR,ABBEVILLE COUNTY,ZIEBACH COUNTY,0,0.00,2636,None,None
3,state,VARCHAR,AK,WY,0,0.00,55,None,None
4,initialFloodHazardBoundaryMap,DATE,1970-02-08,2020-05-29,5368,21.36,894,1975-04-12 02:57:46.666667,None
5,initialFloodInsuranceRateMap,DATE,1970-02-03,2069-10-01,1151,4.58,1919,1990-10-17 13:37:03.524997,None
6,currentlyEffectiveMapDate,VARCHAR,(All Zone D),12/31/82(M),284,1.13,1844,None,None
7,regularEmergencyProgramDate,VARCHAR,01/01/03,12/31/97,2340,9.31,3631,None,None
8,tribal,BOOLEAN,false,true,0,0.00,2,None,None
9,participatingInNFIP,BOOLEAN,false,true,0,0.00,2,None,None


raw_disasters (29 columns, 70,599 rows)


,column_name,column_type,min,max,null_count,null_percentage,approx_unique,avg,std
0,id,VARCHAR,00014251-608a-4fd2-80f7-014c04cdbf02,ffff0a9d-c37e-4d6d-ad34-2f4e9cbcb969,0,0.00,71371,None,None
1,disasterNumber,SMALLINT,1,5679,0,0.00,5307,2740.976288615986,1447.705828695752
2,state,VARCHAR,AK,WY,0,0.00,59,None,None
3,femaDeclarationString,VARCHAR,DR-1-GA,FM-5679-CA,0,0.00,5260,None,None
4,declarationType,VARCHAR,DR,FM,0,0.00,3,None,None
5,declarationDate,DATE,1953-05-02,2026-10-27,0,0.00,3612,2006-07-20 18:40:35.08265,None
6,fyDeclared,SMALLINT,1953,2027,0,0.00,76,2006.2228360175072,15.280114142535478
7,incidentType,VARCHAR,Biological,Winter Storm,0,0.00,26,None,None
8,declarationTitle,VARCHAR,0419 FIRE,ZOGG FIRE,0,0.00,2530,None,None
9,ihProgramDeclared,BOOLEAN,false,true,0,0.00,2,None,None


In [50]:
# Step 2b. Generate the summary statistics for each columns in 
for name, df in other_tables.items():
    print(name, df.shape)
    display(df.describe(include="all").T)

acs (5160, 10)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
NAME,5160,5160,Census Tract 2.01; Alachua County; Florida,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
median_household_income,5160.0,NaN,NaN,NaN,-14011404.341085,95884965.875568,-666666666.0,50165.5,66262.5,87150.0,250001.0
median_home_value,5160.0,NaN,NaN,NaN,-25789460.062209,129371872.799654,-666666666.0,187650.0,273400.0,379025.0,2000001.0
median_age,5160.0,NaN,NaN,NaN,-10594271.207907,83378567.49659,-666666666.0,36.6,42.2,50.6,81.9
occupied_housing_units,5160.0,NaN,NaN,NaN,1618.883915,747.554907,0.0,1115.0,1537.5,2035.0,7737.0
owner_occupied_units,5160.0,NaN,NaN,NaN,1082.543411,646.532576,0.0,640.0,997.0,1428.0,6944.0
total_population,5160.0,NaN,NaN,NaN,4192.738178,2091.172479,0.0,2782.75,3925.0,5319.75,23706.0
state,5160.0,NaN,NaN,NaN,12.0,0.0,12.0,12.0,12.0,12.0,12.0
county,5160.0,NaN,NaN,NaN,71.692442,35.938525,1.0,39.0,86.0,99.0,133.0
tract,5160.0,NaN,NaN,NaN,87157.662597,229403.987266,100.0,6402.75,14405.0,40202.0,990200.0


cpi (957, 2)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
observation_date,957,957,1947-01-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CPIAUCSL,956.0,NaN,NaN,NaN,125.806279,91.038627,21.48,32.895,110.3,202.825,336.298


dgs (16918, 2)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
observation_date,16918,16918,1962-01-02,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DGS10,16197.0,NaN,NaN,NaN,5.803432,2.92571,0.52,3.9,5.38,7.52,15.84


... (2 outputs left out)

mortgage (2902, 2)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
observation_date,2902,2902,1971-04-02,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MORTGAGE30US,2902.0,NaN,NaN,NaN,7.679866,3.175315,2.65,5.5325,7.22,9.21,18.63


unrate (946, 2)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
observation_date,946,946,1948-01-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UNRATE,945.0,NaN,NaN,NaN,5.651323,1.703381,2.5,4.3,5.4,6.7,14.8


#### 2.3 Correct the dtypes of each data for all datasets

In [58]:
# Setup the mapping of the dtypes to SQL types based on the data dictionary
type_map = {
    "text": "VARCHAR",
    "date": "DATE",
    "smallint": "INTEGER",
    "integer": "INTEGER",
    "bigint": "BIGINT",
    "decimal": "DOUBLE",
    "boolean": "BOOLEAN",
    "uuid": "VARCHAR",
    "datetimez": "DATE",
    "decimal(8,2)": "DOUBLE",
    "decimal(9,1)": "DOUBLE",
    "int": "FLOAT",
    "float": "FLOAT",
    "string": "VARCHAR",
    "datetime": "DATE",
    "decimal(6,1)": "DOUBLE",
    "decimal(12,2)": "DOUBLE",
    "TIMESTAMP WITH TIME ZONE": "DATE",
    "DECIMAL(8,2)": "DOUBLE",
    "SMALLINT": "INTEGER",
    "DECIMAL(9,1)": "DOUBLE",
    
}

# Map the target type to each column of policies
policies_dict["target_type"] = policies_dict["type"].map(type_map)
claim_dict["target_type"] = claim_dict["type"].map(type_map)
disaster_dict["target_type"] = disaster_dict["type"].map(type_map)
census_dict["target_type"] = census_dict["type"].map(type_map)
community_dict["target_type"] = community_dict["type"].map(type_map)

# Check whether there are data types that we're not able to map
dictionary = {
    "policies dictionary": policies_dict,
    "claim dictionary": claim_dict,
    "disaster dictionary": disaster_dict,
    "census dictionary": census_dict,
    "community dictionary": community_dict
}

for name, df in dictionary.items():
    print(name, df.shape)
    display(df.loc[df["target_type"].isna(),["name", "type"]])

policies dictionary (87, 10)


,name,type


claim dictionary (84, 10)


,name,type


disaster dictionary (29, 10)


,name,type


census dictionary (6, 5)


,name,type


community dictionary (18, 10)


,name,type


In [63]:
# Checking the dtypes of policies
current = con.sql("DESCRIBE raw_policies").df()[["column_name", "column_type"]]
check = current.merge(policies_dict[["name","target_type"]],
                      left_on="column_name", right_on="name", how="left")
mismatch = check[check["column_type"] != check["target_type"]]
print("Policies")
display(mismatch)

Policies


,column_name,column_type,name,target_type
1,asOfDate,TIMESTAMP WITH TIME ZONE,asOfDate,DATE
2,baseFloodElevation,"DECIMAL(8,2)",baseFloodElevation,DOUBLE
3,basementEnclosureCrawlspaceType,SMALLINT,basementEnclosureCrawlspaceType,INTEGER
7,crsClassCode,SMALLINT,crsClassCode,INTEGER
13,federalPolicyFee,SMALLINT,federalPolicyFee,INTEGER
15,hfiaaSurcharge,SMALLINT,hfiaaSurcharge,INTEGER
17,locationOfContents,SMALLINT,locationOfContents,INTEGER
18,lowestAdjacentGrade,"DECIMAL(8,2)",lowestAdjacentGrade,DOUBLE
19,lowestFloorElevation,"DECIMAL(8,2)",lowestFloorElevation,DOUBLE
21,numberOfFloorsInInsuredBuilding,SMALLINT,numberOfFloorsInInsuredBuilding,INTEGER


Since the dtypes are correct based on the matching performed for policies, no conversion of dtypes is required. 

In [68]:
# Checking the dtypes of claims
current = con.sql("DESCRIBE raw_claims").df()[["column_name", "column_type"]]
check = current.merge(claim_dict[["name","target_type"]],
                      left_on="column_name", right_on="name", how="left")
mismatch = check[check["column_type"] != check["target_type"]]
print("Claims")
display(mismatch)

Claims


,column_name,column_type,name,target_type
1,asOfDate,TIMESTAMP WITH TIME ZONE,asOfDate,DATE
2,basementEnclosureCrawlspaceType,SMALLINT,basementEnclosureCrawlspaceType,INTEGER
3,policyCount,SMALLINT,policyCount,INTEGER
4,crsClassCode,SMALLINT,crsClassCode,INTEGER
9,baseFloodElevation,"DECIMAL(6,1)",baseFloodElevation,DOUBLE
12,locationOfContents,SMALLINT,locationOfContents,INTEGER
13,lowestAdjacentGrade,"DECIMAL(6,1)",lowestAdjacentGrade,DOUBLE
14,lowestFloorElevation,"DECIMAL(6,1)",lowestFloorElevation,DOUBLE
15,numberOfFloorsInTheInsuredBuilding,SMALLINT,numberOfFloorsInTheInsuredBuilding,INTEGER
17,obstructionType,SMALLINT,obstructionType,INTEGER


Since the dtypes are correct based on the matching performed for claims, no conversion of dtypes is required.

In [70]:
# Checking the dtypes of community
current = con.sql("DESCRIBE raw_community").df()[["column_name", "column_type"]]
check = current.merge(community_dict[["name","target_type"]],
                      left_on="column_name", right_on="name", how="left")
mismatch = check[check["column_type"] != check["target_type"]]
print("Community")
display(mismatch)

Community


,column_name,column_type,name,target_type
6,currentlyEffectiveMapDate,VARCHAR,currentlyEffectiveMapDate,DATE
7,regularEmergencyProgramDate,VARCHAR,regularEmergencyProgramDate,DATE
12,classRating,VARCHAR,classRating,INTEGER
17,lastRefresh,TIMESTAMP,lastRefresh,DATE


The variables, 'currentlyEffectiveMapDate' and 'regularEmergencyProgramDate' have wrong types. This must be converted from VARCHAR to DATE types.

In [71]:
# Look at the values first
for c in ["currentlyEffectiveMapDate", "regularEmergencyProgramDate", "classRating"]:
    print(c)
    display(con.sql(f"""
        SELECT "{c}" AS value, COUNT(*) AS n
        FROM raw_community
        GROUP BY 1 ORDER BY 2 DESC LIMIT 15
    """).df())

currentlyEffectiveMapDate


,value,n
0,(NSFHA),1618
1,None,283
2,09/25/09,231
3,09/28/07,199
4,09/29/10,144
5,07/31/24,142
6,04/03/12,129
7,12/17/10,129
8,09/26/08,127
9,09/26/14,124


regularEmergencyProgramDate


,value,n
0,None,2339
1,04/30/84,533
2,09/01/86,178
3,09/04/85,166
4,08/19/85,149
5,09/29/78,144
6,09/24/84,139
7,09/27/85,136
8,09/18/85,135
9,09/28/79,114


classRating


,value,n
0,None,23347
1,7,427
2,8,353
3,10,270
4,6,258
5,5,231
6,9,215
7,3,11
8,2,10
9,4,9


In [74]:
n_fail = con.execute(f"""
    SELECT COUNT(*) FROM raw_community
    WHERE ...
""").fetchone()[0]

ParserException: Parser Error: syntax error at or near "..."

LINE 3:     WHERE ...
                  ^^^

In [75]:
for c in ["currentlyEffectiveMapDate", "regularEmergencyProgramDate"]:
    print(c)
    display(con.sql(f"""
        SELECT "{c}" AS value, COUNT(*) AS n
        FROM raw_community
        WHERE NOT regexp_matches(TRIM("{c}"), '^\\d{{2}}/\\d{{2}}/\\d{{2}}$')
        GROUP BY 1 ORDER BY 2 DESC
    """).df())

currentlyEffectiveMapDate


,value,n
0,(NSFHA),1618
1,11/13/26(>),119
2,12/17/10(M),74
3,04/03/12(M),64
4,12/10/26(>),62
...,...,...
723,11/07/24(M),1
724,08/28/19(M),1
725,12/18/86(M),1
726,01/01/87(L),1


regularEmergencyProgramDate


,value,n
0,08/05/14(E),6
1,02/05/01(E),4
2,01/04/01(E),4
3,01/16/01(E),4
4,03/26/97(E),3
...,...,...
154,06/14/02(E),1
155,08/10/15(E),1
156,07/02/96(E),1
157,01/31/03(E),1


In [67]:
# Compare with what's stored now
current = con.sql("DESCRIBE raw_disasters").df()[["column_name", "column_type"]]
check = current.merge(disaster_dict[["name","target_type"]],
                      left_on="column_name", right_on="name", how="left")
mismatch = check[check["column_type"] != check["target_type"]]
print("Community")
display(mismatch)

Community


,column_name,column_type,name,target_type
1,disasterNumber,SMALLINT,disasterNumber,INTEGER
6,fyDeclared,SMALLINT,fyDeclared,INTEGER
24,region,SMALLINT,region,INTEGER
27,lastRefresh,TIMESTAMP,lastRefresh,DATE


In [ ]:
"raw_community", "raw_disasters"

In [ ]:
claim_dict = pd.read_excel(data_dict / "fema_NfipClaims_v3.xlsx")
policies_dict = pd.read_excel(data_dict / "fema_NfipPolicies_v3.xlsx")
disaster_dict = pd.read_excel(data_dict / "fema_DisasterDeclarationsSummaries_v2.xlsx")
census_dict = pd.read_excel(data_dict / "census_acs5_2022_selected.xlsx")
community_dict = pd.read_excel(data_dict / "fema_NfipCommunityStatusBook_v1.xlsx")

#### 2.4 Check whether the categories for string variables are aligned with the allowed values from the data dictionary. 

#### 2.5 Count the nulls of each columns
Flag constant or near-constant columns and columns that are 100% null.

#### 2.6 Check the validity of the dates
Check plausibility: negative premiums or coverage, coverage above NFIP limits, termination before effective date, construction dates in the future or before 1800, loss dates outside 1978 to today.

#### 2.7 Check for any duplicates in the primary keys (unique IDs)
Check uniqueness: duplicate ids, and fully duplicated rows apart from id.

#### 2.8 Check the validity of censusGeoid and zip codes
Check integrity: claim censusGeoid and zip codes that never appear in policies, and state/county mismatches.

#### 2.9 Export the data quality
Save everything into one dq_audit_raw table: a row per column with % missing, % placeholder, % invalid and so on. This table is your evidence that the data is messy. Export it to reports/.